In [ ]:
!pip install workalendar -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.8/5.8 MB 76.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.7/210.7 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 3.1 MB/s eta 0:00:00


In [ ]:
!pip install kaleido==0.2.1 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 9.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from sklearn.model_selection import TimeSeriesSplit
from tqdm import tqdm
from workalendar.europe import Russia

In [ ]:
df = pd.read_csv("df_final+whether.csv", parse_dates=["timestamp"])
houses = [col for col in df.columns if col.startswith("house_")]

In [ ]:
cal = Russia()

def is_holiday(dt):
    if dt.weekday() >= 5:
        return 0
    return int(not cal.is_working_day(dt.date()))

df["is_holiday"] = df["timestamp"].apply(is_holiday)

In [ ]:
def evaluate(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    mape = mean_absolute_percentage_error(y_true, y_pred) * 100
    return {"MAE": round(mae, 3), "RMSE": round(rmse, 3), "MAPE": round(mape, 3)}

horizons = {
    "4h":  8,
    "8h":  16,
    "24h": 48,
    "7d":  336,
    "14d": 672,
    "1m":  1488,
}

tscv = TimeSeriesSplit(n_splits=5)

In [ ]:
#Тест Дики-Фуллера на стационарность:
for house in houses:
    result = adfuller(df[house], autolag="AIC")
    p_value = result[1]
    stationary = "Cтационарный" if p_value < 0.05 else "Нt стационарный"
    print(f"{house}: p-value={p_value} - {stationary}")

house_1: p-value=2.420914762620979e-08 - Cтационарный
house_2: p-value=5.205384595047096e-08 - Cтационарный
house_3: p-value=1.3497897134782826e-07 - Cтационарный
house_4: p-value=1.454082206467882e-06 - Cтационарный
house_5: p-value=4.933211316863707e-05 - Cтационарный
house_6: p-value=6.631504286946498e-06 - Cтационарный
house_7: p-value=4.5477673153629706e-07 - Cтационарный
house_8: p-value=0.00010515507989016415 - Cтационарный


In [ ]:
order = (1, 0, 0)
seasonal_order = (1, 0, 0, 48)
exog_cols = ["temp_c", "humidity", "cloudiness", "is_holiday"]

In [ ]:
# ограничиваем обучающее окно - последние 6 месяцев
TRAIN_WINDOW = 8640  # 6 месяцев * 30 дней * 48 интервалов

n = len(df)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_idx = list(range(train_end - TRAIN_WINDOW, train_end))
val_idx = list(range(train_end, n))

print(f"Train: {len(train_idx)} точек ({df['timestamp'].iloc[train_idx[0]]} - {df['timestamp'].iloc[train_idx[-1]]})")
print(f"Val:   {len(val_idx)} точек")

Train: 8640 точек (2018-10-25 19:30:00 - 2019-04-23 19:00:00)
Val:   10878 точек


In [ ]:
rows = []

for house in tqdm(houses, desc="SARIMAX"):
    for horizon_name, horizon_points in horizons.items():

        train_data = df[house].iloc[train_idx]
        val_data = df[house].iloc[val_idx[:horizon_points]]
        train_exog = df[exog_cols].iloc[train_idx]
        val_exog = df[exog_cols].iloc[val_idx[:horizon_points]]

        model = SARIMAX(
            train_data,
            exog=train_exog,
            order=order,
            seasonal_order=seasonal_order,
            enforce_stationarity=False,
            enforce_invertibility=False
        )
        fitted = model.fit(disp=False, maxiter=200, method="lbfgs")
        forecast = fitted.forecast(steps=horizon_points, exog=val_exog)

        metrics = evaluate(val_data.values, forecast.values)

        rows.append({
            "модель": "SARIMAX",
            "дом": house,
            "горизонт": horizon_name,
            "MAE":  metrics["MAE"],
            "RMSE": metrics["RMSE"],
            "MAPE": metrics["MAPE"],
        })

        print(f"{house} {horizon_name}: MAE={metrics['MAE']:.2f}")

df_sarimax = pd.DataFrame(rows)
df_sarimax.to_csv("results_sarimax.csv", index=False)

SARIMAX:   0%|          | 0/8 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling th

house_1 4h: MAE=11.29


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the

house_1 8h: MAE=8.23


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the

...

house_8 7d: MAE=10.46


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the supported classes of index.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: An unsupported index was provided. As a result, forecasts cannot be generated. To use the model for forecasting, use one of the

house_8 14d: MAE=13.93


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
SARIMAX: 100%|██████████| 8/8 [2:49:49<00:00, 1273.69s/it]

house_8 1m: MAE=16.82


In [ ]:
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[f"Горизонт {h}" for h in horizons.keys()],
    vertical_spacing=0.1,
    horizontal_spacing=0.08
)

for i, (horizon_name, horizon_points) in enumerate(horizons.items()):
    row = i // 2 + 1
    col = i % 2 + 1

    val_data = df[house].iloc[val_idx[:horizon_points]]
    val_exog = df[exog_cols].iloc[val_idx[:horizon_points]]
    timestamps = df["timestamp"].iloc[val_idx[:horizon_points]]

    forecast = fitted.forecast(steps=horizon_points, exog=val_exog)

    fig.add_trace(go.Scatter(
        x=timestamps, y=val_data.values,
        mode="lines", name="факт",
        line=dict(color="#1f77b4", width=1.2, shape="hv"),
        showlegend=(i == 0)
    ), row=row, col=col)

    fig.add_trace(go.Scatter(
        x=timestamps, y=forecast.values,
        mode="lines", name="прогноз SARIMAX",
        line=dict(color="#d62728", width=1.2, shape="hv"),
        showlegend=(i == 0)
    ), row=row, col=col)

    fig.update_xaxes(title_text="Дата", row=row, col=col, title_font=dict(size=9))
    fig.update_yaxes(title_text="кВт", row=row, col=col, title_font=dict(size=9))

fig.update_layout(
    title="SARIMAX: Прогнозные и фактические значения по всем горизонтам прогноза (дом 1)",
    width=700, height=900,
    font=dict(size=10),
    margin=dict(l=50, r=20, t=40, b=40),
    legend=dict(font=dict(size=9))
)
fig.show()
fig.write_image("29_sarimax_forecast_all_horizons.png", scale=2)

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning:

No supported index is available. Prediction results will be given with an integer index beginning at `start`.

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: FutureWarning:

No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning:

No supported index is available. Prediction results will be given with an integer index beginning at `start`.

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: FutureWarning:

No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning:

No supported index is av

График наглядно показывает слабость SARIMAX на длинных горизонтах - прогноз (красный) сильно расходится с фактом (синий) уже на 7д и полностью теряется на 14д и 1м.
На коротких горизонтах (4h, 8h, 24h) модель улавливает общий тренд, но амплитуда заниженная - это следствие ограниченного обучающего окна

In [ ]:
df_sarimax

,модель,дом,горизонт,MAE,RMSE,MAPE
0,SARIMAX,house_1,4h,11.291,12.016,12.890
1,SARIMAX,house_1,8h,8.234,9.497,13.399
2,SARIMAX,house_1,24h,5.382,6.896,10.692
3,SARIMAX,house_1,7d,25.113,30.395,44.606
4,SARIMAX,house_1,14d,32.183,36.355,61.348
5,SARIMAX,house_1,1m,38.019,41.620,71.345
6,SARIMAX,house_2,4h,2.371,2.821,7.006
7,SARIMAX,house_2,8h,3.383,4.031,17.389
8,SARIMAX,house_2,24h,3.466,4.047,20.375
9,SARIMAX,house_2,7d,9.066,11.230,40.537
